# Temporary M5 region-mask branch validation

Test exactly one existing model/region/period/CIM cell using the pushed feature branch, without overwriting production classification paths. The notebook writes only to a unique `_VALIDATION` GCS object; remove that object manually after reviewing it. Requires Brazil MONITOR_01 access unless the setup cell is adapted.

Open this notebook from the feature branch in Colab; no merge is required. Do not set `RUN_TEST=True` until the four selectors below identify a suitable tile and the staging destination is acceptable.

In [ ]:
import os, subprocess, sys
REPO_DIR = '/content/brazil-fire-region-test'
BRANCH = 'fix/v04-region-only-classification'
if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/mapbiomas/brazil-fire.git', REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', 'FETCH_HEAD'], check=True)
CORE = os.path.join(REPO_DIR, 'network/catalog/development/version_04/src/core')
sys.path.insert(0, CORE)
print(subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD'], text=True).strip())
# Colab dependencies; install GDAL bindings as in the country notebooks.
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'gdal-bin', 'python3-gdal'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'earthengine-api', 'google-cloud-storage', 'gcsfs', 'rasterio', 'numpy<2', 'shapely>=2', 'pyproj>=3.6', 'scipy', 'tqdm'], check=True)

## Configure the one-tile test
Set the selectors from an existing M5 model and COG period. Choose a cell intersecting the region with valid source pixels. The first run authenticates using the account already authorized for the Brazil project/bucket.

In [ ]:
COUNTRY = 'brazil'
CAMPAIGN = 'MONITOR_01'
MODEL_ID = ''       # required: model folder/id in this campaign
REGION_NAME = ''    # required: exact region name in the campaign regions asset
PERIOD = ''         # e.g. '2025_08'; must have source COGs
CELL_ID = ''        # exact CIM cell name, e.g. one returned by the preflight below
RUN_TEST = False    # explicit write gate; leave False until selectors are checked

from M0_auth_config import set_global_opts, authenticate, CONFIG
set_global_opts(
    country=COUNTRY, campaign=CAMPAIGN, sensor='sentinel2', periodicity='monthly',
    mosaic_methods=['minnbr', 'minnbr_buffer'], personal_task_flag='MONITOR', language='pt',
    gcs_bucket='mapbiomas-fire',
    gcs_library_images_prefix=f'sudamerica/{COUNTRY}/CATALOG_01',
    gcs_campaigns_prefix=f'sudamerica/{COUNTRY}/CATALOG_01',
    gee_project=f'mapbiomas-{COUNTRY}',
    gee_library_images_prefix=f'projects/mapbiomas-{COUNTRY}/assets/FIRE/CATALOG_01',
    gee_campaigns_prefix=f'projects/mapbiomas-{COUNTRY}/assets/FIRE/CATALOG_01',
    asset_regions=f'projects/mapbiomas-{COUNTRY}/assets/FIRE/CATALOG_01/{CAMPAIGN}/AUXILIARY/regiones_fuego_{COUNTRY}_v1',
)
if RUN_TEST:
    assert all([MODEL_ID, REGION_NAME, PERIOD, CELL_ID]), 'Set all four selectors before enabling RUN_TEST.'
    authenticate()
    from M0_auth_config import _get_fs
    from M5_classifier import _get_region_cells, _get_region_geometry
    assert any(c['name'] == CELL_ID for c in _get_region_cells(REGION_NAME)), 'CELL_ID is not an intersecting region cell.'
    print('Preflight passed; GCS:', CONFIG['bucket'], 'region:', REGION_NAME, 'cell:', CELL_ID, 'period:', PERIOD)
else:
    print('Dry/preflight only. Set selectors, inspect the target, then explicitly set RUN_TEST=True.')

In [ ]:
if RUN_TEST:
    import uuid, tempfile, rasterio, numpy as np, ee
    from M5_inference import load_model_from_gcs, build_band_paths, classify_cell_with_cogs
    from M0_auth_config import _gcs_models_base, get_temp_dir, _gcs_download
    from M5_spatial import clip_geometry_to_cell, rasterize_region_mask
    from M5_workplan import gcs_full, gcs_classifications_path
    from M5_classifier import _get_region_geometry
    fs = _get_fs()
    model_dir = f"gs://{CONFIG['bucket']}/{_gcs_models_base()}/{MODEL_ID}"
    predict_fn, meta, bands_config, norm_stats, band_order = load_model_from_gcs(model_dir, fs)
    band_paths = build_band_paths(bands_config, PERIOD)
    missing = [p for p in band_paths.values() if not fs.exists(f'gs://{p}')]
    assert not missing, f'Missing source COG(s): {missing}'
    region_geojson = _get_region_geometry(REGION_NAME)
    cim = ee.FeatureCollection('projects/mapbiomas-workspace/AUXILIAR/cim-world-1-250000')
    cell_geojson = cim.filter(ee.Filter.eq('name', CELL_ID)).first().geometry().getInfo()
    staging_key = f"{gcs_classifications_path()}/_VALIDATION/M5_region_mask/{MODEL_ID}/{REGION_NAME}/{CELL_ID}_{PERIOD}_{uuid.uuid4().hex}.tif"
    staging_path = gcs_full(staging_key)
    result = classify_cell_with_cogs(
        CELL_ID, predict_fn, bands_config, norm_stats, staging_path, band_paths, band_order,
        region_geometry=region_geojson, logger=print,
    )
    assert result is not None, 'No valid pixels in this cell/region/period; choose another test cell.'
    local_path = os.path.join(get_temp_dir('tiles'), f'validation_{uuid.uuid4().hex}.tif')
    _gcs_download(staging_path, local_path)
    with rasterio.open(local_path) as src:
        prob = src.read(1)
        clipped, _ = clip_geometry_to_cell(region_geojson, cell_geojson, src.crs)
        inside = rasterize_region_mask(clipped, prob.shape, src.transform)
    outside_classified = int(((prob != -9999) & ~inside).sum())
    classified_pixels = int((prob != -9999).sum())
    assert outside_classified == 0, f'{outside_classified} pixels outside the region were classified.'
    assert classified_pixels == result['total_pixels'], 'Raster count differs from M5 valid-pixel statistics.'
    print({'PASS': True, 'classified_pixels': classified_pixels, 'outside_region_pixels': outside_classified, 'staging_gcs_object': staging_path})
    print('Review the isolated staging object, then remove it manually; this notebook never overwrites production tiles or replaces GEE assets.')